# Recurrence Index

Nino 1+2 sea-surface temperature, relation to ICEN/ONI, and the recurrence index by canton and province (Ecuador, 2015-2025).

Inputs: processed ENSO series and the hazard event table. Outputs: recurrence tables in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Efficient reader for GeoPackage layers
import pyogrio

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step5_recurrence_index", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("recurrencia iniciada — configuración cargada desde %s", CONFIG_PATH)

# First year of the analysis window
YEAR_MIN = config["date_range"]["start_year"]
# Last year of the analysis window
YEAR_MAX = config["date_range"]["end_year"]

## 1. Load raw Nino 1+2 sea-surface temperature

In [ ]:
# Path to the raw NOAA/CPC monthly sea-surface temperature file
nino_path = resolve_path(config, "raw_dir", base_dir=Path.cwd()) / "ersst5.nino.mth.91-20.ascii"

# Explicit column names, because the source header repeats 'ANOM'
COLS = ["year", "month", "nino12_sst", "nino12_anom",
        "nino3_sst", "nino3_anom", "nino4_sst", "nino4_anom",
        "nino34_sst", "nino34_anom"]
# Read the fixed-width text file, skipping its header row
nino_raw = pd.read_csv(nino_path, sep=r"\s+", skiprows=1, names=COLS)
# Log how many rows were loaded and their year range
logger.info("Archivo Niño 1+2/3/4/3.4 cargado: %d filas (%d-%d)", len(nino_raw), nino_raw.year.min(), nino_raw.year.max())

# Keep only the Nino 1+2 columns plus the year/month join keys
nino12 = nino_raw[["year", "month", "nino12_sst", "nino12_anom"]].copy()

# Restrict the series to the analysis window
nino12_2015_2025 = nino12[(nino12.year >= YEAR_MIN) & (nino12.year <= YEAR_MAX)].reset_index(drop=True)
# Check that the window has exactly 12 months per year, no gaps
assert len(nino12_2015_2025) == (YEAR_MAX - YEAR_MIN + 1) * 12, "deben ser 132 meses completos (2015-2025), sin huecos"
# Log the size of the clipped series
logger.info("Niño 1+2 recortado a %d-%d: %d meses, sin huecos", YEAR_MIN, YEAR_MAX, len(nino12_2015_2025))

## 2. Quarterly moving average

In [ ]:
# Sort chronologically so rolling windows are correct
nino12_2015_2025 = nino12_2015_2025.sort_values(["year", "month"]).reset_index(drop=True)
# Centered 3-month moving average, comparable to the ICEN/ONI running quarters
nino12_2015_2025["nino12_anom_trim"] = (
    nino12_2015_2025["nino12_anom"].rolling(window=3, center=True, min_periods=3).mean()
)
# Count edge months without a full 3-month window
n_sin_trim = nino12_2015_2025["nino12_anom_trim"].isna().sum()
# Log how many edge months have no moving average
logger.info("Media móvil trimestral de Niño 1+2 calculada; %d meses en el borde sin trimestre completo", n_sin_trim)

## 3. Merge with the monthly ENSO series

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the monthly ENSO index series (ICEN and ONI)
enso_monthly = pd.read_parquet(processed_dir / "enso_index_monthly.parquet")
# Log the number of monthly ENSO rows loaded
logger.info("enso_index_monthly cargado: %d filas", len(enso_monthly))

# Left-join the Nino 1+2 series onto the ENSO series by year and month
enso_comparado = enso_monthly.merge(
    nino12_2015_2025[["year", "month", "nino12_sst", "nino12_anom", "nino12_anom_trim"]],
    on=["year", "month"], how="left",
)
# Check the join neither lost nor duplicated months
assert len(enso_comparado) == len(enso_monthly), "la unión no debe perder ni duplicar meses"
# Check every month has a Nino 1+2 value
assert enso_comparado["nino12_anom"].isna().sum() == 0, "todo mes 2015-2025 debe tener SST de Niño 1+2 (archivo cubre 1950-2026)"
# Log the size of the combined series
logger.info("Serie ENSO combinada (ICEN + ONI + Niño 1+2): %d meses, sin nulos en Niño 1+2", len(enso_comparado))

## 4. Relation between ICEN, ONI and Nino 1+2

In [ ]:
# Correlation between ICEN and ONI
corr_icen_oni = enso_comparado["ICEN"].corr(enso_comparado["ONI"])
# Correlation between ICEN and raw Nino 1+2 anomaly
corr_icen_nino12 = enso_comparado["ICEN"].corr(enso_comparado["nino12_anom_trim"])
# Correlation between ONI and raw Nino 1+2 anomaly
corr_oni_nino12 = enso_comparado["ONI"].corr(enso_comparado["nino12_anom_trim"])

# Log the three correlations
logger.info(
    "Correlaciones 2015-2025 (n=%d, trimestral donde aplica): ICEN-ONI=%.3f | ICEN-Niño1+2=%.3f | ONI-Niño1+2=%.3f",
    enso_comparado["nino12_anom_trim"].notna().sum(), corr_icen_oni, corr_icen_nino12, corr_oni_nino12,
)
# Print ICEN vs ONI correlation
print(f"ICEN vs. ONI:            r = {corr_icen_oni:.3f}")
# Print ICEN vs Nino 1+2 correlation
print(f"ICEN vs. Niño 1+2 (crudo, trimestral): r = {corr_icen_nino12:.3f}")
# Print ONI vs Nino 1+2 correlation
print(f"ONI  vs. Niño 1+2 (crudo, trimestral): r = {corr_oni_nino12:.3f}")

## 5. Operational definition of an El Nino month

In [ ]:
# Recurrence index settings from the config
REC_CFG = config["recurrence_index"]
# ENSO magnitude settings from the config
MAG_CFG = config["enso_magnitude_index"]
# Category prefix that marks an El Nino month
ENSO_PREFIX = REC_CFG["enso_event_categories_prefix"]
# Mapping from ENSO category to an ordinal value between 0 and 1
CATEGORY_VALUES = MAG_CFG["category_values"]

# Flag months classified as El Nino by ICEN only (kept for traceability)
enso_comparado["es_evento_enso_icen_solo"] = enso_comparado["icen_category"].str.startswith(ENSO_PREFIX)
# Flag months classified as El Nino by ONI only
enso_comparado["es_evento_enso_oni_solo"] = enso_comparado["oni_category"].str.startswith(ENSO_PREFIX)

# Operational flag: El Nino according to ICEN or ONI
enso_comparado["es_evento_enso"] = enso_comparado["es_evento_enso_icen_solo"] | enso_comparado["es_evento_enso_oni_solo"]

# Classify each month by which index detected the El Nino
def _tipo_enso(row):
    # Both indices agree
    if row["es_evento_enso_icen_solo"] and row["es_evento_enso_oni_solo"]:
        # Label for months flagged by both indices
        return "Ambos"
    # Only ICEN flags the month
    if row["es_evento_enso_icen_solo"]:
        # Label for coastal El Nino
        return "Niño Costero"
    # Only ONI flags the month
    if row["es_evento_enso_oni_solo"]:
        # Label for basin-wide El Nino
        return "Niño"
    # Neither index flags the month
    return "No ENSO"

# Apply the classification to every month
enso_comparado["tipo_enso"] = enso_comparado.apply(_tipo_enso, axis=1)

# Map ICEN categories to ordinal magnitude values
icen_ord = enso_comparado["icen_category"].map(CATEGORY_VALUES)
# Map ONI categories to ordinal magnitude values
oni_ord = enso_comparado["oni_category"].map(CATEGORY_VALUES)
# Check every ICEN category was mapped
assert icen_ord.isna().sum() == 0, "categoria ICEN sin mapear en config.yaml enso_magnitude_index.category_values"
# Check every ONI category was mapped
assert oni_ord.isna().sum() == 0, "categoria ONI sin mapear en config.yaml enso_magnitude_index.category_values"
# Monthly magnitude is the stronger of the two indices
enso_comparado["magnitud_enso_combinada"] = pd.concat([icen_ord, oni_ord], axis=1).max(axis=1)

# Number of El Nino months in the window
N_MESES_ENSO = int(enso_comparado["es_evento_enso"].sum())
# Total number of months in the window
N_MESES_TOTAL = len(enso_comparado)
# Log the El Nino month count
logger.info("Meses 'evento ENSO' AMPLIADO (ICEN o ONI El Niño), 2015-2025: %d de %d", N_MESES_ENSO, N_MESES_TOTAL)
# Log the breakdown by detecting index
logger.info("Desglose tipo_enso: %s", enso_comparado['tipo_enso'].value_counts().to_dict())
# Check the El Nino month count matches the verified value
assert N_MESES_ENSO == 54, "debe coincidir con la union ICEN+ONI verificada (38 ICEN + 16 solo-ONI netos = 54)"

## 6. Do events concentrate in El Nino months?

In [ ]:
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)

# Count events per year-month
eventos_por_mes = he.groupby(["year", "month"]).size().rename("n_eventos").reset_index()
# Align the counts to all 132 months so months without events appear as zero
todos_meses = enso_comparado[["year", "month", "es_evento_enso"]].merge(
    eventos_por_mes, on=["year", "month"], how="left"
)
# Fill months with no events with zero
todos_meses["n_eventos"] = todos_meses["n_eventos"].fillna(0).astype(int)
# Check the reindexed total equals the number of events
assert todos_meses["n_eventos"].sum() == len(he), "la suma reindexada debe igualar el total de eventos"

# Mean events per month during El Nino months
tasa_elnino = todos_meses.loc[todos_meses.es_evento_enso, "n_eventos"].mean()
# Mean events per month outside El Nino months
tasa_no_elnino = todos_meses.loc[~todos_meses.es_evento_enso, "n_eventos"].mean()
# Ratio of the two rates
razon = tasa_elnino / tasa_no_elnino if tasa_no_elnino > 0 else float("nan")

# Log both rates and the ratio
logger.info(
    "Tasa media de eventos/mes — El Niño: %.1f (n=%d meses) | No-El-Niño: %.1f (n=%d meses) | razón: %.2fx",
    tasa_elnino, todos_meses.es_evento_enso.sum(), tasa_no_elnino, (~todos_meses.es_evento_enso).sum(), razon,
)
# Print the El Nino monthly rate
print(f"Eventos/mes en meses El Niño:     {tasa_elnino:.1f}  (n={todos_meses.es_evento_enso.sum()} meses)")
# Print the non-El Nino monthly rate
print(f"Eventos/mes en meses No-El-Niño:  {tasa_no_elnino:.1f}  (n={(~todos_meses.es_evento_enso).sum()} meses)")
# Print the ratio
print(f"Razón: {razon:.2f}x")

## 7. Recurrence index by canton and province

In [ ]:
# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the join did not change the number of events
assert len(he) == n_antes
# Check every event received an official name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0

# Set of (year, month) pairs that are El Nino months
meses_enso_set = set(
    map(tuple, enso_comparado.loc[enso_comparado.es_evento_enso, ["year", "month"]].values)
)
# Flag each event as occurring in an El Nino month or not
he["mes_es_enso"] = he.apply(lambda r: (r["year"], r["month"]) in meses_enso_set, axis=1)

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]


# Compute the recurrence index for an administrative level
def calcular_recurrencia(df: pd.DataFrame, admin_cols: list[str], n_meses_enso: int) -> pd.DataFrame:
    """Compute the recurrence index per administrative unit, by hazard type and for all hazards combined.
    
    Args:
        df: Hazard events with the boolean column 'mes_es_enso'.
        admin_cols: Columns that identify the administrative level.
        n_meses_enso: Total number of El Nino months in the window.
    
    Returns:
        Table with El Nino months with at least one event, the recurrence index (0-1)
        and the raw event count in El Nino months, per unit and view.
    """
    # Helper that computes recurrence for one view of the data
    def _por_vista(sub: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
        # Keep only events that fall in El Nino months
        enso_sub = sub[sub.mes_es_enso]
        # Number of distinct El Nino months with at least one event, per unit
        meses_con_evento = (
            enso_sub.groupby(cols + ["year", "month"]).size().reset_index()
            .groupby(cols).size().rename("meses_enso_con_evento")
        )
        # Raw count of events in El Nino months, kept for traceability
        n_eventos = enso_sub.groupby(cols).size().rename("n_eventos_en_meses_enso")
        # Combine both measures into one table
        out = pd.concat([meses_con_evento, n_eventos], axis=1).reset_index()
        # Recurrence is the share of El Nino months with at least one event
        out["recurrence_index"] = out["meses_enso_con_evento"] / n_meses_enso
        # Return the per-unit table
        return out

    # Recurrence broken down by hazard type
    por_hazard = _por_vista(df, admin_cols + ["hazard_type"])
    # Recurrence for all hazards combined
    combinado = _por_vista(df, admin_cols)
    # Label the combined view as 'TODOS' (all hazards)
    combinado["hazard_type"] = "TODOS"

    # Universe of all units that have any historical event
    universo = df[admin_cols].drop_duplicates()
    # Add units with no event in El Nino months so they appear with zero
    combinado = universo.merge(combinado, on=admin_cols, how="left")
    # Re-apply the combined label after the join
    combinado["hazard_type"] = "TODOS"
    # Loop over the two count columns
    for col in ["meses_enso_con_evento", "n_eventos_en_meses_enso"]:
        # Replace missing counts with zero
        combinado[col] = combinado[col].fillna(0).astype(int)
    # Replace missing recurrence with zero
    combinado["recurrence_index"] = combinado["recurrence_index"].fillna(0.0)

    # Stack the per-hazard and combined views
    result = pd.concat([por_hazard, combinado], ignore_index=True)
    # Check recurrence stays within 0 and 1
    assert result["recurrence_index"].between(0, 1).all(), "recurrence_index fuera de [0,1]"
    # Return the final table
    return result


# Log the start of the canton-level calculation
logger.info("Calculando Índice de Recurrencia a nivel cantón...")
# Recurrence at canton level
rec_canton = calcular_recurrencia(he, CANTON_COLS, N_MESES_ENSO)
# Log the start of the province-level calculation
logger.info("Calculando Índice de Recurrencia a nivel provincia...")
# Recurrence at province level
rec_provincia = calcular_recurrencia(he, PROVINCE_COLS, N_MESES_ENSO)
# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(rec_canton), len(rec_provincia))

## 8. Validation

In [ ]:
# Check no canton exceeds the total number of El Nino months
assert (rec_canton["meses_enso_con_evento"] <= N_MESES_ENSO).all(), "meses_enso_con_evento no puede superar N_MESES_ENSO"
# Check no province exceeds the total number of El Nino months
assert (rec_provincia["meses_enso_con_evento"] <= N_MESES_ENSO).all(), "ídem, provincia"

# Number of cantons with historical events
n_cantones_esperados = he[CANTON_COLS].drop_duplicates().shape[0]
# Number of cantons present in the combined view
n_cantones_en_recurrencia = (rec_canton["hazard_type"] == "TODOS").sum()
# Check every canton appears in the combined view
assert n_cantones_esperados == n_cantones_en_recurrencia, \
    f"faltan cantones en la vista TODOS: {n_cantones_esperados} esperados, {n_cantones_en_recurrencia} presentes"

# Number of provinces with historical events
n_provincias_esperadas = he[PROVINCE_COLS].drop_duplicates().shape[0]
# Number of provinces present in the combined view
n_provincias_en_recurrencia = (rec_provincia["hazard_type"] == "TODOS").sum()
# Check every province appears in the combined view
assert n_provincias_esperadas == n_provincias_en_recurrencia, "faltan provincias en la vista TODOS"

# Count cantons whose recurrence is zero
n_en_cero = ((rec_canton.hazard_type == "TODOS") & (rec_canton.recurrence_index == 0)).sum()
# Log the coverage check
logger.info(
    "Cobertura completa verificada: %d/%d cantones, %d/%d provincias en vista TODOS (%d cantones con recurrence_index=0)",
    n_cantones_en_recurrencia, n_cantones_esperados, n_provincias_en_recurrencia, n_provincias_esperadas, n_en_cero,
)

# Ten cantons with the highest recurrence
top10 = rec_canton[rec_canton.hazard_type == "TODOS"].sort_values("recurrence_index", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official", "canton_name_official", "meses_enso_con_evento",
             "recurrence_index", "n_eventos_en_meses_enso"]].to_string(index=False))

## 9. Export

In [ ]:
# Tables to export
tablas = {
    "recurrence_index_canton": rec_canton,
    "recurrence_index_provincia": rec_provincia,
    "enso_index_monthly_extended": enso_comparado,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")